In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
!curl -A "Mozilla/5.0" -L -o /content/development.mp4 https://raw.githubusercontent.com/intel-iot-devkit/sample-videos/master/person-bicycle-car-detection.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5889k  100 5889k    0     0  16.9M      0 --:--:-- --:--:-- --:--:-- 16.9M


In [4]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 10.2 MB/s eta 0:00:00


In [5]:
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import yaml

from IPython.display import Video, display
from ultralytics import YOLO
from ultralytics.utils import ROOT

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
DAY_02_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_02"
)

RAW_VIDEO_PATH = Path("/content/development.mp4")
DEFAULT_TRACKS_PATH = DAY_02_DIR / "tracks.csv"
DEFAULT_VIDEO_PATH = DAY_02_DIR / "bytetrack_baseline.mp4"

DAY_04_DIR = Path(
    "/content/drive/MyDrive/Vision/vision_unit_03/outputs/day_04"
)

DAY_04_DIR.mkdir(parents=True,exist_ok=True)

BUFFER60_VIDEO_PATH = DAY_04_DIR / "bytetrack_buffer60.mp4"
BUFFER60_TRACKS_PATH = DAY_04_DIR / "tracks_buffer60.csv"
CUSTOM_TRACKER_PATH = DAY_04_DIR / "bytetrack_buffer60.yaml"

MODEL_NAME = "yolo26n.pt"
IMAGE_SIZE = 640
CONFIDENCE_THRESHOLD = 0.10
NMS_IOU_THRESHOLD = 0.70

TARGET_CLASS_NAMES = {
    "person",
    "bicycle",
    "car",
}

DEVICE = (0 if torch.cuda.is_available() else "cpu")

In [ ]:
def read_video_metadata(path):
    capture = cv2.VideoCapture(str(path))

    metadata = {
        "width": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "height": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "fps": float(capture.get(cv2.CAP_PROP_FPS)),
        "frames": int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    }

    capture.release()
    return metadata


raw_metadata = read_video_metadata(RAW_VIDEO_PATH)

default_video_metadata = read_video_metadata(DEFAULT_VIDEO_PATH)

metadata_comparison_df = pd.DataFrame(
    [
        {
            "source": "raw_video",
            **raw_metadata,
        },
        {
            "source": "day2_baseline",
            **default_video_metadata,
        },
    ]
)

display(metadata_comparison_df)


same_video_structure = (
    raw_metadata["width"] == default_video_metadata["width"]
    and
    raw_metadata["height"] == default_video_metadata["height"]
    and
    raw_metadata["frames"] == default_video_metadata["frames"]
    and
    abs(raw_metadata["fps"] - default_video_metadata["fps"])
    < 0.1
)

,source,width,height,fps,frames
0,raw_video,768,432,12.0,647
1,day2_baseline,768,432,12.0,647


**Custom ByteTrack configuration**